# **Choosing the G2P: All Candidates on the Same Test Split**

The drug-name pipeline needs a G2P for Filipino words, and this notebook chooses it. Instead of
taking whichever model scores lowest, it compares one candidate from each main family of G2P,
rule-based, statistical and neural, each built for Filipino or covering it. All of them are scored
under the same conditions, and a decision rule fixed in advance makes the choice.

| Candidate | Family | Why it's included | Notebook | Folds |
| --- | --- | --- | --- | --- |
| Epitran `tgl-Latn` (Mortensen et al., 2018) | rule-based | a widely used rule-based G2P, with hand-written Tagalog rules | `Epitran_eval.ipynb` | scored only |
| WFST (Phonetisaurus) | statistical | the joint-sequence model used by Aquino et al. (2019) for Tagalog, trained on this data | `Wik_eval.ipynb` | retrained on each fold |
| `filipino-byt5-g2p` (Marqueses et al., 2026) | neural, existing | a recent neural G2P built for Filipino, used as released | `ByT5_eval.ipynb` | scored only |
| CharsiuG2P fine-tuned (Zhu et al., 2022) | neural, trained on this data | a multilingual neural G2P that covers Tagalog, fine-tuned on the WFST's training data | `ByT5_finetune.ipynb` | retrained on each fold |

The two neural candidates answer different questions. `filipino-byt5-g2p` shows how an existing
Filipino model does here, but it learned from other labels (sentences, with stress) and has seen
many of the test words. The fine-tuned CharsiuG2P is the controlled comparison: it learns from
exactly the same data as the WFST model, so a difference between them comes from the model, not the
data. Both are ByT5 models, so they're called by these names rather than "ByT5".

Every candidate is held to the same conditions:

- **The same test split**, `dicts/cwik_test.tsv` (2,171 words), written by `Wik_eval.ipynb`. It is
  never used to choose anything: settings, input formats and epochs are chosen on the validation
  split in each candidate's notebook.
- **The same scoring code**, `compute_per_wer()`, and a prediction for every test word.
- **The same 10 folds**, those of `Wik_eval.ipynb`. A candidate that learns from the data is
  retrained on each fold. The others can't be retrained the same way, so each fold is only scored:
  their spread shows how stable they are, but it isn't a cross-validated estimate.

A candidate is included once its notebook has written its predictions to `output/`. The
fine-tuned CharsiuG2P needs a GPU, so it may be missing.

**Decision rule**, fixed before looking at the results: the candidate with the lowest test PER is
chosen, provided that its lead over each other candidate is statistically significant and that it
also has the lowest mean PER across the folds.

**The process at a glance**

1. **Score** every candidate on the test split and collect its fold PERs.
2. **Test** whether the differences from the best candidate are significant.
3. **Choose** the G2P.
4. **State** what the comparison does and doesn't show.

In [1]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

sys.dont_write_bytecode = True  # no __pycache__ in ../utils
sys.path.append(os.path.abspath("../utils"))
from tsv_metrics import compute_per_wer
from alignment import align_trxn

current_dir = Path().resolve()
dicts_dir   = current_dir / "dicts"
output_dir  = current_dir / "output"

CANDIDATES = {   # name: (kind, predictions, fold PERs, how the folds were scored)
    "Epitran":               ("rule-based",  "epitran_output.txt", "epitran_cv.tsv", "scored only"),
    "WFST":                  ("statistical", "cwik_output.txt",    "cwik_cv.tsv",    "retrained"),
    "filipino-byt5-g2p":     ("neural",      "byt5_output.txt",    "byt5_cv.tsv",    "scored only"),
    "CharsiuG2P fine-tuned": ("neural",      "byt5ft_output.txt",  "byt5ft_cv.tsv",  "retrained"),
}


def read_output(path):
    """Read a word TAB phones file into {word: [phones]}."""
    preds = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) >= 2:
                preds[parts[0]] = parts[1].split()
    return preds


ref = read_output(dicts_dir / "cwik_test.tsv")
preds = {name: read_output(output_dir / out) for name, (_, out, _, _) in CANDIDATES.items()
         if (output_dir / out).is_file()}
print(f"Test words: {len(ref)}")
print("Candidates found:", ", ".join(preds))
print("Missing:", ", ".join(n for n in CANDIDATES if n not in preds) or "none")

Test words: 2171
Candidates found: Epitran, WFST, filipino-byt5-g2p
Missing: CharsiuG2P fine-tuned


## 1. Score every candidate

PER and WER on the test split, and the mean and standard deviation of the fold PERs.

The WFST model has the lowest test PER, **0.81%**, ahead of `filipino-byt5-g2p` (1.07%) and
Epitran (4.80%). It also has the lowest mean fold PER, **0.99%** (± 0.06%), with the smallest
spread, even though it is the only one of the three that is retrained on each fold.

In [2]:
rows = []
for name, hyp in preds.items():
    kind, _, cv_file, cv_how = CANDIDATES[name]
    m = compute_per_wer(ref, hyp)
    assert m["n_words"] == len(ref), f"{name}: scored {m['n_words']} of {len(ref)} test words"
    cv = pd.read_csv(output_dir / cv_file, sep="\t")["per"] if (output_dir / cv_file).is_file() else pd.Series(dtype=float)
    rows.append({
        "Candidate": name, "Kind": kind,
        "Test PER (%)": round(m["per"] * 100, 2), "Test WER (%)": round(m["wer"] * 100, 2),
        "Fold PER mean (%)": round(cv.mean() * 100, 2) if len(cv) else None,
        "Fold PER std (%)": round(cv.std(ddof=0) * 100, 2) if len(cv) else None,
        "Folds": len(cv), "Folds were": cv_how,
    })

scores = pd.DataFrame(rows).sort_values("Test PER (%)").reset_index(drop=True)
best = scores.loc[0, "Candidate"]
scores

,Candidate,Kind,Test PER (%),Test WER (%),Fold PER mean (%),Fold PER std (%),Folds,Folds were
0,WFST,statistical,0.81,5.85,0.99,0.06,10,retrained
1,filipino-byt5-g2p,neural,1.07,6.77,1.21,0.13,10,scored only
2,Epitran,rule-based,4.80,28.79,5.23,0.25,10,scored only


## 2. Test the differences

Each candidate is compared with the best one, word by word, on the same test words:

- **Paired bootstrap.** The test words are resampled with replacement 10,000 times (fixed seed),
  and the PER difference is recomputed on each resample. The 2.5th and 97.5th percentiles give a
  95% confidence interval for the difference. If it excludes 0, the difference is significant.
- **Wilcoxon signed-rank test** on the per-word edit distances, as in Marqueses et al. (2026).
  Words where both candidates make the same number of edits are left out, the test's standard
  treatment of ties. The p-values are corrected for the number of comparisons with the
  Holm–Bonferroni method, at α = 0.05.

Both differences are significant. Against Epitran the gap is large: **3.99** percentage points
(95% CI 3.63 to 4.35), with Epitran worse on 559 words and better on 62 (Holm-corrected
p < 10⁻⁸³). Against `filipino-byt5-g2p` it is small but significant: **0.26** points (95% CI 0.04
to 0.50), with `filipino-byt5-g2p` worse on 102 words and better on 83 (Holm-corrected p = 0.031).

In [3]:
words = list(ref)
n_ref = np.array([len(ref[w]) for w in words])
edits = {name: np.array([align_trxn(ref[w], hyp[w])[2] for w in words]) for name, hyp in preds.items()}

rng = np.random.default_rng(42)
B = 10_000
samples = [rng.integers(0, len(words), len(words)) for _ in range(B)]

rows = []
for name in scores["Candidate"][1:]:
    diff = edits[name] - edits[best]                     # > 0 where the candidate is worse
    boot = np.array([diff[s].sum() / n_ref[s].sum() for s in samples]) * 100
    stat, p = wilcoxon(edits[name], edits[best])
    rows.append({
        "Candidate": name, "vs": best,
        "PER difference (pp)": round(diff.sum() / n_ref.sum() * 100, 2),
        "95% CI (pp)": f"[{np.percentile(boot, 2.5):.2f}, {np.percentile(boot, 97.5):.2f}]",
        "Words worse": int((diff > 0).sum()), "Words better": int((diff < 0).sum()),
        "Wilcoxon W": stat, "p": p,
    })

tests = pd.DataFrame(rows)
# Holm-Bonferroni: the i-th smallest p is multiplied by (m - i), and the result kept monotonic.
order = tests["p"].argsort().to_numpy()
m = len(tests)
adjusted = np.maximum.accumulate([min(1.0, (m - i) * tests["p"].iloc[j]) for i, j in enumerate(order)])
tests.loc[tests.index[order], "p (Holm)"] = adjusted
tests["Significant"] = tests["p (Holm)"] < 0.05
tests

,Candidate,vs,PER difference (pp),95% CI (pp),Words worse,Words better,Wilcoxon W,p,p (Holm),Significant
0,filipino-byt5-g2p,WFST,0.26,"[0.04, 0.50]",102,83,7154.0,3.073390e-02,3.073390e-02,True
1,Epitran,WFST,3.99,"[3.63, 4.35]",559,62,14074.0,3.390896e-84,6.781792e-84,True


## 3. Choose the G2P

The decision rule from the top of the notebook, applied to the tables above.

The WFST model meets every condition of the rule: the lowest test PER, the lowest mean fold PER
and a significant lead over both other candidates. It is chosen as the G2P.

The fine-tuned CharsiuG2P isn't included yet. Once `ByT5_finetune.ipynb` has been run on a GPU,
rerunning this notebook adds it to every table and applies the same rule.

In [4]:
cv_best = scores.dropna(subset=["Fold PER mean (%)"]).sort_values("Fold PER mean (%)").iloc[0]["Candidate"]
all_significant = bool(tests["Significant"].all())

print(f"Lowest test PER          : {best}")
print(f"Lowest mean fold PER     : {cv_best}")
print(f"Lead significant vs all  : {all_significant}")
print()
if best == cv_best and all_significant:
    print(f"Chosen: {best}")
else:
    print("No candidate meets every condition of the decision rule; see the tables above.")

Lowest test PER          : WFST
Lowest mean fold PER     : WFST
Lead significant vs all  : True

Chosen: WFST


## 4. What the comparison does and doesn't show

- **The references are Wiktionary's phonemic transcriptions.** A candidate is scored against this
  dictionary's conventions, which are the ones the drug-name pipeline uses. Part of Epitran's gap
  is convention rather than wrong sounds: it writes *sy* and *ny* as single phones (`ʃ`, `ɲ`) and
  reads spelled glides as glides (`Epitran_eval.ipynb`, section 7). Its biggest gap, the glottal
  stop, is a real one: it writes `ʔ` only at the start of a word.
- **`filipino-byt5-g2p` has the advantage of seen words.** 38.5% of the test words appear in its
  published training sentences. On the words it hasn't seen, its PER is 1.17% against the WFST
  model's 0.71% (`ByT5_eval.ipynb`, section 8), so the overall gap understates the WFST model's
  lead, if anything.
- **"Scored only" folds aren't cross-validation.** For Epitran and `filipino-byt5-g2p`, the fold
  PERs show how stable each model is on different words, not how it would do if trained on them.
- **One language and one dictionary.** The result says which G2P best reproduces this Tagalog
  dictionary. It doesn't rank the systems in general, and each was run with the settings
  documented in its own notebook.